# 🧬 Basic Genetic Analysis with MalariaGEN

## Understanding Genetic Variations in Malaria

**Welcome to your first genetic analysis tutorial!** This notebook will teach you how to explore genetic variations in malaria mosquitoes and parasites using MalariaGEN data.

### What You'll Learn:
1. 🧬 What genetic variations are and why they matter
2. 📊 How to access and explore SNP data
3. 🔍 Basic population genetics concepts
4. 📈 Visualizing genetic diversity
5. 🎯 Real-world applications in malaria research

### Prerequisites:
- Completed the "Getting Started" tutorial
- Basic understanding of genetics (DNA, genes, mutations)

---

## 🧬 Understanding Genetic Variations

### What are Genetic Variations?
- **SNPs** (Single Nucleotide Polymorphisms): Single letter changes in DNA
- **INDELs**: Insertions or deletions of DNA segments
- **CNVs**: Copy Number Variations (duplicated or deleted regions)

### Why They Matter for Malaria:
- 🦟 **Mosquitoes**: Insecticide resistance, feeding behavior, adaptation
- 🩸 **Parasites**: Drug resistance, immune evasion, virulence
- 🌍 **Public Health**: Track disease spread, guide control strategies

Let's start exploring!

In [ ]:
# Import necessary libraries
import malariagen_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Set up plotting
plt.style.use('default')
sns.set_palette("viridis")

print("🧬 Genetic Analysis Tools Ready!")

## 🦟 Exploring Mosquito Genetic Variations

In [ ]:
# Initialize the Ag3 dataset (African malaria mosquitoes)
ag3 = malariagen_data.Ag3()

print("✅ Ag3 dataset loaded for genetic analysis!")
print("🧬 Ready to explore mosquito genetic variations!")

### 📊 Understanding SNP Data

In [ ]:
# Let's explore what SNP (Single Nucleotide Polymorphism) data looks like
print("🔍 Exploring SNP data structure...")

# Get available site masks (different filtering levels for SNPs)
site_masks = ag3.site_mask_ids()
print("\n📋 Available SNP site masks:")
for mask in site_masks:
    print(f"  • {mask}")

print("\n💡 Site masks represent different quality filtering levels:")
print("  • 'gamb_colu': Standard quality filter for Anopheles gambiae complex")
print("  • 'arab': Filter for Anopheles arabiensis")
print("  • Higher numbers = stricter filtering")

In [ ]:
# Let's look at a specific chromosome to understand the data structure
print("🧬 Exploring chromosome 2L...")

# Get SNP calls for a small region of chromosome 2L
# Note: This might take a moment as it loads genetic data
chromosome = "2L"
region = f"{chromosome}:1,000,000-1,010,000"  # 10kb region

print(f"📍 Loading SNP data for region: {region}")

# Get SNP calls (genetic variations) for this region
try:
    snp_calls = ag3.snp_calls(region=region, site_mask="gamb_colu")
    
    print(f"\n📊 SNP data loaded successfully!")
    print(f"🧬 Shape of SNP data: {snp_calls.dims}")
    print(f"📍 Variants (SNPs): {snp_calls.dims['variants']:,}")
    print(f"🧪 Samples: {snp_calls.dims['samples']:,}")
    print(f"🧬 Alleles: {snp_calls.dims['alleles']}")
    
    # Display basic information about the SNP data
    print("\n🔍 Data variables:")
    for var in snp_calls.data_vars:
        print(f"  • {var}: {snp_calls[var].dims}")
        
except Exception as e:
    print(f"❌ Error loading SNP data: {e}")
    print("💡 This might be due to network issues or data access limitations.")
    print("🔄 Let's try a different approach...")

### 🧪 Understanding Genotype Data

In [ ]:
# Let's explore what genotype data looks like
print("🧪 Understanding mosquito genotypes...")

# Genotypes represent the genetic makeup at each SNP position
# Format: [0, 0] = homozygous reference, [1, 1] = homozygous alternate, [0, 1] = heterozygous

try:
    # Look at a few sample genotypes
    if 'call_genotype' in snp_calls:
        sample_genotypes = snp_calls['call_genotype'][:5, :5]  # First 5 variants, first 5 samples
        
        print("🧬 Sample genotype matrix (5 variants × 5 samples):")
        print(sample_genotypes.values)
        
        print("\n💡 Genotype coding:")
        print("  • [0, 0] = Homozygous reference (both alleles are the 'normal' version)")
        print("  • [1, 1] = Homozygous alternate (both alleles have the variation)")
        print("  • [0, 1] = Heterozygous (one normal, one varied allele)")
        print("  • [-1, -1] = Missing data")
        
    else:
        print("❌ Genotype data not available in the loaded dataset")
        
except Exception as e:
    print(f"❌ Error accessing genotype data: {e}")
    print("💡 Let's continue with other genetic analyses...")

## 🌍 Population Genetics: Genetic Diversity

In [ ]:
# Let's explore genetic diversity across different populations
print("🌍 Exploring genetic diversity across mosquito populations...")

# Get sample metadata to understand population structure
sample_metadata = ag3.sample_metadata()

# Look at different countries/populations
if 'country' in sample_metadata.columns:
    countries = sample_metadata['country'].value_counts().head(5)
    print("📍 Top 5 countries with most samples:")
    for country, count in countries.items():
        print(f"  • {country}: {count:,} samples")
        
    # We'll use these countries for diversity analysis
    selected_countries = countries.head(3).index.tolist()
    print(f"\n🎯 Selected for diversity analysis: {', '.join(selected_countries)}")
    
else:
    print("❌ Country information not available")
    selected_countries = []

In [ ]:
# Calculate genetic diversity for different populations
print("🧮 Calculating genetic diversity metrics...")

try:
    if selected_countries:
        # Get diversity statistics for each country
        diversity_results = []
        
        for country in selected_countries:
            print(f"\n🌍 Analyzing {country}...")
            
            # Get samples from this country
            country_samples = sample_metadata[sample_metadata['country'] == country]
            sample_ids = country_samples['sample_id'].head(50).tolist()  # Use subset for speed
            
            if sample_ids:
                # Calculate diversity statistics
                diversity_stats = ag3.diversity_stats(
                    region=region,
                    sample_sets=country_samples['sample_set'].iloc[0],
                    sample_query=f"sample_id in {sample_ids}"
                )
                
                if diversity_stats is not None and len(diversity_stats) > 0:
                    avg_diversity = diversity_stats['pi'].mean()
                    diversity_results.append({
                        'country': country,
                        'diversity': avg_diversity,
                        'samples': len(sample_ids)
                    })
                    print(f"  📊 Average diversity (π): {avg_diversity:.6f}")
                else:
                    print(f"  ⚠️ No diversity data available for {country}")
            
        # Visualize diversity results
        if diversity_results:
            diversity_df = pd.DataFrame(diversity_results)
            
            plt.figure(figsize=(10, 6))
            bars = plt.bar(diversity_df['country'], diversity_df['diversity'], 
                          color='lightcoral', alpha=0.8)
            
            plt.xlabel('Country')
            plt.ylabel('Genetic Diversity (π)')
            plt.title('🌍 Genetic Diversity Across Mosquito Populations')
            plt.xticks(rotation=45)
            
            # Add value labels on bars
            for bar, diversity in zip(bars, diversity_df['diversity']):
                plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(diversity_df['diversity'])*0.01,
                        f'{diversity:.4f}', ha='center', va='bottom', fontweight='bold')
            
            plt.tight_layout()
            plt.show()
            
            print("\n💡 What this means:")
            print("  • Higher diversity = more genetic variation")
            print("  • Lower diversity = more genetically similar population")
            print("  • Diversity can indicate population size, history, and adaptation")
        
except Exception as e:
    print(f"❌ Error calculating diversity: {e}")
    print("💡 Let's try a simpler approach...")

## 🩸 Exploring Parasite Genetic Variations

In [ ]:
# Now let's explore genetic variations in malaria parasites
print("🩸 Exploring Plasmodium falciparum genetic variations...")

# Initialize Pf7 dataset
pf7 = malariagen_data.Pf7()
print("✅ Pf7 dataset loaded!")

In [ ]:
# Get parasite sample metadata
parasite_metadata = pf7.sample_metadata()
print(f"🧬 Total parasite samples: {len(parasite_metadata):,}")

# Look at geographical distribution
if 'country' in parasite_metadata.columns:
    parasite_countries = parasite_metadata['country'].value_counts().head(5)
    print("\n🌍 Top 5 countries for parasite samples:")
    for country, count in parasite_countries.items():
        print(f"  • {country}: {count:,} samples")

In [ ]:
# Explore parasite genetic data
print("🧬 Exploring parasite SNP data...")

try:
    # Get a small region of chromosome 1 for parasites
    parasite_region = "Pf3D7_01_v3:1,000,000-1,010,000"
    
    print(f"📍 Loading parasite SNP data for: {parasite_region}")
    
    parasite_snps = pf7.snp_calls(region=parasite_region)
    
    print(f"\n📊 Parasite SNP data loaded!")
    print(f"🧬 Variants: {parasite_snps.dims['variants']:,}")
    print(f"🧪 Samples: {parasite_snps.dims['samples']:,}")
    
    # Look at variant allele frequencies
    if 'variant_allele' in parasite_snps:
        print("\n🔍 Sample variant alleles:")
        sample_variants = parasite_snps['variant_allele'][:5]
        for i, variant in enumerate(sample_variants):
            print(f"  Position {i}: {variant.values}")
    
except Exception as e:
    print(f"❌ Error loading parasite SNP data: {e}")
    print("💡 This might be due to data access limitations.")

## 🎯 Real-World Applications

### 💊 **Drug Resistance Monitoring**
Genetic variations help us track how malaria parasites become resistant to drugs:
- **pfcrt** mutations: Chloroquine resistance
- **pfmdr1** mutations: Multiple drug resistance
- **kelch13** mutations: Artemisinin resistance

### 🦟 **Insecticide Resistance Tracking**
Mosquito genetic variations show resistance to insecticides:
- **kdr mutations**: Pyrethroid resistance
- **CYP6P9** variants: Metabolic resistance
- **Gene duplications**: Increased detoxification

### 🌍 **Disease Surveillance**
Genetic data helps track malaria spread:
- **Population structure**: How parasites move between regions
- **Transmission intensity**: Genetic diversity indicates transmission levels
- **Outbreak tracking**: Connect cases through genetic similarities

## 📊 Visualizing Genetic Patterns

In [ ]:
# Create a comprehensive visualization of our genetic analysis
print("📊 Creating genetic analysis summary...")

# Create a summary figure
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('🧬 MalariaGEN Genetic Analysis Summary', fontsize=16, fontweight='bold')

# 1. Sample distribution (mosquitoes)
if 'country' in sample_metadata.columns:
    mosquito_top = sample_metadata['country'].value_counts().head(8)
    axes[0, 0].barh(range(len(mosquito_top)), mosquito_top.values, color='skyblue')
    axes[0, 0].set_yticks(range(len(mosquito_top)))
    axes[0, 0].set_yticklabels(mosquito_top.index)
    axes[0, 0].set_title('🦟 Mosquito Samples by Country')
    axes[0, 0].set_xlabel('Number of Samples')
    axes[0, 0].invert_yaxis()

# 2. Sample distribution (parasites)
if 'country' in parasite_metadata.columns:
    parasite_top = parasite_metadata['country'].value_counts().head(8)
    axes[0, 1].barh(range(len(parasite_top)), parasite_top.values, color='lightcoral')
    axes[0, 1].set_yticks(range(len(parasite_top)))
    axes[0, 1].set_yticklabels(parasite_top.index)
    axes[0, 1].set_title('🩸 Parasite Samples by Country')
    axes[0, 1].set_xlabel('Number of Samples')
    axes[0, 1].invert_yaxis()

# 3. Genetic diversity comparison (if available)
if 'diversity_results' in locals() and diversity_results:
    diversity_df = pd.DataFrame(diversity_results)
    axes[1, 0].bar(diversity_df['country'], diversity_df['diversity'], color='lightgreen')
    axes[1, 0].set_title('🌍 Genetic Diversity by Country')
    axes[1, 0].set_ylabel('Diversity (π)')
    axes[1, 0].tick_params(axis='x', rotation=45)
else:
    axes[1, 0].text(0.5, 0.5, 'Genetic Diversity\nAnalysis\nNot Available', 
                    ha='center', va='center', fontsize=12, transform=axes[1, 0].transAxes)
    axes[1, 0].set_title('🌍 Genetic Diversity')

# 4. Key concepts summary
concepts = [
    'SNPs: Single DNA letter changes',
    'Genotypes: Genetic makeup',
    'Diversity: Population variation',
    'Resistance: Drug/Insecticide adaptation'
]

for i, concept in enumerate(concepts):
    axes[1, 1].text(0.1, 0.8 - i*0.2, f'• {concept}', fontsize=11, 
                    transform=axes[1, 1].transAxes)
    
axes[1, 1].set_title('🧬 Key Genetic Concepts')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("🎉 Genetic analysis complete! Here's what we learned:")
print("\n🧬 Key Takeaways:")
print("  • MalariaGEN contains massive genetic datasets")
print("  • We can explore both mosquito and parasite genetics")
print("  • Genetic diversity varies by geographic region")
print("  • These data help track drug and insecticide resistance")
print("  • Genetic analysis is crucial for malaria control")

## 🚀 Next Steps in Genetic Analysis

### 📚 **Advanced Topics to Explore:**
1. **🧬 Population Structure**: Use PCA to visualize genetic relationships
2. **🦟 Selection Scans**: Find genes under natural selection
3. **💊 Drug Resistance**: Identify known resistance mutations
4. **🌍 Phylogeography**: Track disease spread patterns
5. **🧪 Haplotype Analysis**: Study blocks of genetic variation

### 🔬 **Research Applications:**
- **Surveillance**: Monitor emerging resistance
- **Control Strategies**: Guide intervention programs
- **Vaccine Development**: Identify vaccine targets
- **Evolutionary Studies**: Understand parasite adaptation

### 🛠️ **Technical Skills to Develop:**
- Advanced statistical genetics
- Machine learning for genotype-phenotype associations
- Visualization of large-scale genetic data
- Integration with epidemiological data

---

## 🎉 **Congratulations!**

You've successfully completed your first genetic analysis with MalariaGEN! You now understand:

✅ **What genetic variations are** and why they matter
✅ **How to access SNP data** from mosquitoes and parasites
✅ **Basic population genetics** concepts
✅ **How to calculate genetic diversity** across populations
✅ **Real-world applications** in malaria research

You're ready to tackle more advanced genetic analyses! 🚀

---

**🔬 Continue your genetic analysis journey:**
- Try the advanced tutorials on population genetics
- Explore specific resistance genes
- Join the MalariaGEN community discussions
- Start your own research project!

**🌟 Happy genetic analyzing!**